<a href="https://colab.research.google.com/github/jeffrey96158/Vertical-Federated-Learning-without-explicit-ID-Matching/blob/main/Vertical_Federated_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Package

In [ ]:
# pytorch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torchvision.datasets as dset 
import torchvision.transforms as transforms
import torch.utils.data as data

# others
from tqdm.notebook import tqdm as tqdm
import argparse
from sklearn.metrics import f1_score
import random

# import matplotlib.pyplot as plt
# import numpy as np

# Download Dataset (MNIST)
MNIST is a data set of handwrite number

In [ ]:
trans = transforms.Compose([transforms.ToTensor()]) 
train_set = dset.MNIST(root='.', train=True, download=True ,transform=trans)
test_set = dset.MNIST(root='.', train=False,transform=trans)

  0%|          | 0/9912422 [00:00<?, ?it/s]

Extracting ./MNIST/raw/train-images-idx3-ubyte.gz to ./MNIST/raw



  0%|          | 0/28881 [00:00<?, ?it/s]

Extracting ./MNIST/raw/train-labels-idx1-ubyte.gz to ./MNIST/raw



  0%|          | 0/1648877 [00:00<?, ?it/s]

Extracting ./MNIST/raw/t10k-images-idx3-ubyte.gz to ./MNIST/raw



  0%|          | 0/4542 [00:00<?, ?it/s]

Extracting ./MNIST/raw/t10k-labels-idx1-ubyte.gz to ./MNIST/raw



# Arguments

In [ ]:
parser = argparse.ArgumentParser(description='FL with PyTorch MNIST')
parser.add_argument('--no-cuda', action='store_true', default=False, 
                    help='disables CUDA training')
parser.add_argument('--seed', type=int, default=50, metavar='S',
                    help='random seed (default: 50)')
parser.add_argument('--epochs', type=int, default=1, metavar='N',
                    help='number of epochs to train (default: 1)')
parser.add_argument('--lr', type=float, default=0.1, metavar='LR',
                    help='learning rate (default: 0.1)')
parser.add_argument('--log-interval', type=int, default=10, metavar='N',
                    help='how many batches to wait before logging training status')
parser.add_argument('--batch-size', type=int, default=50, metavar='N',
                    help='input batch size for training (default: 50)')
parser.add_argument('--test-batch-size', type=int, default=50, metavar='N',
                    help='input batch size for testing (default: 50)')
parser.add_argument('--num-participants', type=int, default=10, metavar='NP',
                    help='number of participants (default: 10)')
parser.add_argument('--randomorder_rounds', type=int, default=1, metavar='NP',
                    help='rounds for Random Order (default: 1)')
parser.add_argument('--asynchronous_rounds', type=int, default=1, metavar='NP',
                    help='rounds for Asynchronous (default: 1)')
args = parser.parse_args(args=[])

# Class Definition

In [ ]:
## Initialization
# Control Seed
# CUDA is a GPU core for parrallel computing
torch.manual_seed(args.seed)

# Select Device
use_cuda = not args.no_cuda and torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else 'cpu')

if use_cuda:
    print("Using CUDA!")
    torch.cuda.manual_seed(args.seed)
else:
    print('Not using CUDA!!!')

Using CUDA!


# Network

In [ ]:
#使用linear regression model
#連續使用3次linear regression
class network(nn.Module):
    def __init__(self):
        super(network,self).__init__()
        self.L1 = nn.Linear(784,128)
        self.L2 = nn.Linear(128,64)
        self.output = nn.Linear(64,10)
    def forward(self , x):
        x = F.relu(self.L1(x))
        x = F.relu(self.L2(x))
        x = self.output(x)  #若是loss用crossentropy 他最後一層會自己用softmax
        return x
        
class client_network(nn.Module):
    def __init__(self):
        super(client_network,self).__init__()
        self.L1 = nn.Linear(784,128)
        self.L2 = nn.Linear(128,64)
    def forward(self , x):
        x = F.relu(self.L1(x))
        x = F.relu(self.L2(x))
        return x

class server_network(nn.Module):
    def __init__(self):
        super(server_network,self).__init__()
        self.output = nn.Linear(64,10)
    def forward(self , x):
        x = self.output(x)  #若是loss用crossentropy 他最後一層會自己用softmax
        return x

#Client

In [ ]:
class Participant():
    def __init__(self):
        global device
        self.local_network = client_network().to(device)
        self.loss_fn = nn.CrossEntropyLoss()
        self.lr = args.lr
        self.optimizer = torch.optim.SGD(params=self.local_network.parameters(torch.tensor(1.0, requires_grad=True)), lr = self.lr)
        self.loss_history = []
        self.testloss_history = []
        self.accu_history = []
        #download grandient_buffer
        self.gradient_buffer = dict()
    def Upload(self, net_out, GlobalServer):
        GlobalServer.ServerFP(net_out)

    def LocalTraining(self, train_dataset, GlobalServer, epochs = 1, DownUp_cmd = "DU"):

        for e in range(epochs):
            epoch_loss_sum = 0

            pbar = tqdm(enumerate(train_dataset), total=len(train_dataset))
            for batch_idx, (data, label) in pbar:
                data, label = data.to(device), label.to(device)
                batch_size = data.shape[0]
                data = data.view(batch_size,-1) # 把data tensor的shape做改變
                net_out = self.local_network(data) # 把x丟進net去train，得到output
                ##上傳net_out給server
                self.Upload(net_out, GlobalServer)
                #從server下載gradient
                self.Download(GlobalServer)
                self.AddGradient()

                if (batch_idx+1) % args.log_interval == 0:
                    done = (batch_idx+1) * len(data)
                    percentage = 100. * (batch_idx+1) / len(train_dataset)
                    pbar.set_description(f'Train Epoch: {e} [{done:5}/{len(train_dataset.dataset)} ({percentage:3.0f}%)]')

    def LocalTesting(self, test_dataset):
        self.local_network.eval()
        correct = 0
        epoch_loss_sum = 0
        Labels = []
        Predicts = []
        with torch.no_grad():
            for data, label in test_dataset:
                data, label = data.to(device), label.to(device)
                batchsize = data.shape[0]
                data = data.view(batchsize, -1)
                net_out = self.local_network(data)
                loss = self.loss_fn(net_out, label)
                epoch_loss_sum += float(loss.item())

                predict = net_out.data.max(1, keepdim=True)[1]  # get the index of the max log-probability
                correct += predict.eq(label.data.view_as(predict)).sum().item()
                Labels = Labels + label.tolist()
                Predicts = Predicts + predict.tolist()

            epoch_loss_sum /= len(test_dataset.dataset)
            accuracy = 100. * correct / len(test_dataset.dataset)
            self.testloss_history.append(epoch_loss_sum)
            self.accu_history.append(accuracy / 100)
            f1 = f1_score(Labels, Predicts, average = "macro")
            print(f'Test set: Average loss: {epoch_loss_sum:.4f}, Accuracy: {correct}/{len(test_dataset.dataset)} ({accuracy:.2f}%), F1-score: {f1:.4f}')



#Server

In [ ]:
class GlobalServer():
    def __init__(self):
        global device
        self.global_network = server_network().to(device)
        self.gradient_buffer = dict()
        self.temp_gradient_buffer = dict()
        self.loss_history = []
        self.lr = args.lr
        self.optimizer = torch.optim.SGD(params=self.global_network.parameters(torch.tensor(1.0, requires_grad=True)), lr = self.lr)
        self.loss_fn = nn.CrossEntropyLoss()

        # GlobalServer set a dict to preserve net_out
        self.net_out_buffer = dict()
        self.label_buffer = dict()
    def AddGradient(self):
        state_dict_tmp = self.global_network.state_dict()
        for key in state_dict_tmp.keys():
            #print(f'adding {key}')
            state_dict_tmp[key] += self.gradient_buffer[key]
        self.temp_gradient_buffer = self.gradient_buffer.copy()
        self.global_network.load_state_dict(state_dict_tmp)
        self.gradient_buffer.clear()
    def Append_loss(self, loss_sum):
        self.loss_history.append(loss_sum)

    def ServerFP(self, net_out):
        #label = label.to(device)
        server_out = self.global_network(net_out)
        loss = self.loss_fn(net_out,label)
        self.optimizer.zero_grad() # clear gradient
        loss.backward() # back propagation
        #opt.step() # parameter update
        # for layer, param in zip(ln.state_dict().keys(), ln.parameters()):
        #   if (self.gradient_buffer.get(layer) == None): 
        #     self.gradient_buffer[layer] = -1 * param.grad * self.lr
        #   else:
        #     self.gradient_buffer[layer] -= param.grad * self.lr
        self.AddGradient();

#SplitData

In [ ]:
def GenerateParticipant(num = 1):
    P_dict = dict()
    for i in range(num):
        s = 'P'
        s = s + str(i)
        P_dict[s] = Participant()

    return P_dict
    
def SplitData(train_set, test_set, num):
    if len(train_set) % num or len(test_set) % num:
        print(f"len(train_set):{len(train_set)}, len(test_set):{len(test_set)}, can not be indivisible by {num}")
        return
    
    train_split = int(len(train_set) / num)
    test_split = int(len(test_set) / num)

    portions = [train_split] * num
    TrainSet_list = [None] * num
    TrainSet_list = data.random_split(train_set, portions)

    portions = [test_split] * num
    TestSet_list = [None] * num
    TestSet_list = data.random_split(test_set, portions)

    TrainDataSet_dict = dict()
    TestDataSet_dict = dict()

    # mini batch
    for i in range(num):
        s = 'P'
        s = s + str(i)
        TrainDataSet_dict[s] = data.DataLoader(dataset =  TrainSet_list[i], batch_size=args.batch_size, shuffle=True)
        TestDataSet_dict[s] = data.DataLoader(dataset =  TestSet_list[i], batch_size=args.test_batch_size, shuffle=True)

    return TrainDataSet_dict, TestDataSet_dict

## Random Order

In [ ]:
#測試準確率的 test dataset (全部10,000的資料)
test_dataset = data.DataLoader(dataset=test_set, batch_size=args.test_batch_size,shuffle=True)
# 用來檢測accuracy
test_P = GenerateParticipant(1) 

# Dataset and Participant Gernerate
Participant_dict = GenerateParticipant(args.num_participants)
TrainData_dict, TestData_dict = SplitData(train_set, test_set, args.num_participants)

# Participants keys
Participant_keys = list(Participant_dict.keys())

# global server initialization
Server = GlobalServer()

NameError: ignored

# Vertical Federated Learning
Someday we will test accuracy


In [ ]:
for _ in tqdm(range(args.randomorder_rounds)):
    Upload_list = random.sample(Participant_keys, args.num_participants)

    print('Server testing before:')
    test_P["P0"].Download(Server)
    test_P["P0"].LocalTesting(test_dataset)

    for key in Upload_list:
        print('='*10, key, '='*10)
        Participant_dict[key].VLocalTraining(TrainData_dict[key], Server, args.epochs, "DU")
        #print('Server testing after:')
        # test_P["P0"].Download(Server)
        # test_P["P0"].LocalTesting(test_dataset)
    print('Server testing after:')
    test_P["P0"].Download(Server)
    test_P["P0"].LocalTesting(test_dataset)
    
    for key in Upload_list:
        print(f'Participant {key} testing:')
        Participant_dict[key].LocalTesting(test_dataset)